### Automated Search (Optuna) after Hard-Coded Analysis

This automated optimization is performed **after** the preliminary hard-coded parameter analysis conducted in  
`experiments/parameter-two-stage-finding.ipynb`, and is restricted to the development split.

The hard-coded exploration identified a stable performance peak for the two-stage strategy around the following configuration:

- **C** = 1.0  
- **min_rule_purity** = 0.95  
- **min_rule_support** = 40  
- **macro_f1** ≈ 0.721  
- **rule_coverage** ≈ 0.156  

This setting isolates approximately 15–20% of the samples through highly confident deterministic rules, while delegating the remaining ambiguous instances to the probabilistic classifier.

The automated search yields a slightly higher best score:

- **Best F1** ≈ 0.722  
- **Best parameters**:  
  - **C** ≈ 0.97  
  - **min_rule_purity** ≈ 0.93  
  - **min_rule_support** = 30  

The marginal improvement over the hard-coded configuration confirms the presence of a **flat optimum region** rather than a sharp peak. Importantly, the optimal parameters found by the automated search lie in the same neighborhood previously identified manually, thereby **reconfirming the hypotheses of the hard-coded analysis**.

From a modeling perspective, **C** controls the regularization of the linear classifier applied to ambiguous samples, **min_rule_purity** defines the confidence threshold for deterministic rule application, and **min_rule_support** regulates the statistical reliability of such rules. Together, these parameters define a precision–coverage trade-off that is shown to be stable across both manual and automated exploration.


In [ ]:
#Libraries

import pandas as pd
import numpy as np
import re
import optuna

from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score


c:\Users\msist\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
#Paths
DEV_PATH = '../../data/processed/development_processed.csv'

In [ ]:
#Load Data 
df = pd.read_csv(DEV_PATH)

EXPECTED_COLS = {
    "source", "title", "article", "label",
    "text", "n_tokens", "title_len", "article_len", "title_ratio"
}

missing = EXPECTED_COLS - set(df.columns)
assert not missing, f"Missing columns: {missing}"

In [10]:
# Santy check 
df = df.drop(columns=["timestamp"], errors="ignore")

df["source"]  = df["source"].fillna("").astype(str)
df["text"]    = df["text"].fillna("").astype(str)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]
df[NUM_COLS] = (
    df[NUM_COLS]
    .replace([np.inf, -np.inf], 0)
    .fillna(0)
)

In [11]:
#Fetures 
FEATURES = ["source", "text"] + NUM_COLS

X = df[FEATURES]
y = df["label"].astype(int)

In [ ]:
# Straified Fold Optuna
skf = StratifiedKFold(
	n_splits=5,
	shuffle=True,
	random_state=42
)

tr_idx, te_idx = next(skf.split(X, y))

X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
X_te, y_te = X.iloc[te_idx], y.iloc[te_idx]

In [13]:
# One Stage rules and Mining 

def tokenize_for_rules(text):
	return re.findall(r"[a-z0-9_:/\.]+", text.lower())

def mine_pure_rules(texts, labels, min_support, min_purity):
	counts = defaultdict(lambda: Counter())

	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rules = {}
	meta  = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < min_support:
			continue

		best_cls, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= min_purity:
			rules[tok] = best_cls
			meta[tok]  = (purity, total)

	return rules, meta

def apply_rules(texts, rules, meta):
	pred = np.full(len(texts), -1, dtype=int)

	for i, txt in enumerate(texts):
		hits = [t for t in set(tokenize_for_rules(txt)) if t in rules]
		if not hits:
			continue

		hits.sort(
			key=lambda t: (meta[t][0], meta[t][1]),
			reverse=True
		)

		pred[i] = rules[hits[0]]

	return pred


In [14]:
# Model 

def make_model(word_ng_max, char_ng_max, min_df, max_df, C):
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			("w", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, word_ng_max),
				min_df=min_df,
				max_df=max_df,
				sublinear_tf=True,
				max_features=250_000
			), "text"),

			("c", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, char_ng_max),
				min_df=min_df,
				max_df=max_df,
				sublinear_tf=True,
				max_features=300_000
			), "text"),

			("n", StandardScaler(), NUM_COLS),
		],
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf)
	])


In [15]:
def objective(trial):

	# RULE PARAMS 
	min_support = trial.suggest_int("min_rule_support", 20, 80)
	min_purity  = trial.suggest_float("min_rule_purity", 0.90, 0.99)

	# TF-IDF PARAMS 
	word_ng_max = trial.suggest_categorical("word_ng_max", [1, 2])
	char_ng_max = trial.suggest_categorical("char_ng_max", [4, 5])
	min_df      = trial.suggest_categorical("min_df", [2, 3, 5])
	max_df      = trial.suggest_float("max_df", 0.85, 0.95)

	# LR 
	C = trial.suggest_float("C", 0.3, 5.0, log=True)

	#TRAIN MODEL 
	model = make_model(
		word_ng_max,
		char_ng_max,
		min_df,
		max_df,
		C
	)

	model.fit(X_tr, y_tr)
	model_pred = model.predict(X_te)

	# RULE STAGE 
	rules, meta = mine_pure_rules(
		X_tr["text"],
		y_tr,
		min_support,
		min_purity
	)

	rule_pred = apply_rules(X_te["text"], rules, meta)

	final_pred = model_pred.copy()
	mask = rule_pred != -1
	final_pred[mask] = rule_pred[mask]

	return f1_score(y_te, final_pred, average="macro")


In [16]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=25, show_progress_bar=True)

print("\n=-------------- BEST RESULT --------------")
print("Best Macro-F1:", study.best_value)
print("Best Params:")
for k, v in study.best_params.items():
	print(f"  {k}: {v}")

[I 2026-01-22 20:33:30,432] A new study created in memory with name: no-name-39b4200f-ab2d-4610-bd12-f255bcf4033e
Best trial: 0. Best value: 0.710576:   4%|▍         | 1/25 [03:53<1:33:34, 233.93s/it]

[I 2026-01-22 20:37:24,368] Trial 0 finished with value: 0.7105759633240704 and parameters: {'min_rule_support': 61, 'min_rule_purity': 0.960093863882763, 'word_ng_max': 1, 'char_ng_max': 4, 'min_df': 5, 'max_df': 0.9257455833290201, 'C': 1.645154892572002}. Best is trial 0 with value: 0.7105759633240704.


Best trial: 1. Best value: 0.710792:   8%|▊         | 2/25 [11:28<2:19:27, 363.82s/it]

[I 2026-01-22 20:44:59,105] Trial 1 finished with value: 0.7107923154144481 and parameters: {'min_rule_support': 41, 'min_rule_purity': 0.9299794316114647, 'word_ng_max': 2, 'char_ng_max': 5, 'min_df': 5, 'max_df': 0.9478404827018361, 'C': 4.137785727719376}. Best is trial 1 with value: 0.7107923154144481.


Best trial: 2. Best value: 0.714754:  12%|█▏        | 3/25 [17:20<2:11:20, 358.19s/it]

...

[I 2026-01-22 22:41:44,258] Trial 22 finished with value: 0.7157269319474823 and parameters: {'min_rule_support': 20, 'min_rule_purity': 0.9678021141037162, 'word_ng_max': 2, 'char_ng_max': 5, 'min_df': 2, 'max_df': 0.9175270811658778, 'C': 0.5276106203820263}. Best is trial 18 with value: 0.7178359835983689.


Best trial: 18. Best value: 0.717836:  96%|█████████▌| 24/25 [2:15:24<06:38, 398.98s/it]

[I 2026-01-22 22:48:54,889] Trial 23 finished with value: 0.7170953999946913 and parameters: {'min_rule_support': 26, 'min_rule_purity': 0.9505396138236957, 'word_ng_max': 2, 'char_ng_max': 5, 'min_df': 2, 'max_df': 0.9010221505514157, 'C': 0.7562938340515183}. Best is trial 18 with value: 0.7178359835983689.


Best trial: 18. Best value: 0.717836: 100%|██████████| 25/25 [2:23:49<00:00, 345.18s/it]

[I 2026-01-22 22:57:19,933] Trial 24 finished with value: 0.716041003836373 and parameters: {'min_rule_support': 27, 'min_rule_purity': 0.952808743872124, 'word_ng_max': 2, 'char_ng_max': 5, 'min_df': 2, 'max_df': 0.8996599985004212, 'C': 1.270048280325415}. Best is trial 18 with value: 0.7178359835983689.

=-------------- BEST RESULT --------------
Best Macro-F1: 0.7178359835983689
Best Params:
  min_rule_support: 20
  min_rule_purity: 0.984162581589456
  word_ng_max: 2
  char_ng_max: 5
  min_df: 3
  max_df: 0.9077050155274585
  C: 0.8424574689304396
